# PBIX Documenter -- Quickstart with a Real PBIX File

This notebook downloads a publicly available sample PBIX file from Microsoft's
GitHub repository and runs the full PBIX Documenter pipeline against it --
no Streamlit app required.

It is the fastest way to see what the tool actually produces on a real Power BI model.

**What this notebook does:**
1. Downloads the Microsoft "Sales & Returns" sample PBIX file
2. Parses the semantic model using `pbixray`
3. Generates Markdown documentation (without LLM explanations, so no Ollama needed)
4. Optionally adds LLM explanations if Ollama is running
5. Exports the result to an HTML file you can open in your browser

**Prerequisites:**
- `pip install -r ../requirements.txt` (or use `uv pip install -r ../requirements.txt`)
- Ollama is optional -- Section 4 will skip gracefully if it is not running

---

## 1. Download the Sample PBIX File

We use the Microsoft Power BI Desktop Samples repository, which is publicly
available under the MIT licence. The "Sales & Returns" report is a realistic
model with multiple tables, DAX measures, and relationships.

In [ ]:
import requests
import os

download_url = (
    'https://raw.githubusercontent.com/microsoft/powerbi-desktop-samples'
    '/refs/heads/main/Sample%20Reports/Sales%20%26%20Returns%20Sample%20v201912.pbix'
)
pbi_file = 'sample.pbix'

if os.path.exists(pbi_file):
    print(f'File already exists: {pbi_file} ({os.path.getsize(pbi_file) / 1024:.0f} KB) -- skipping download')
else:
    print('Downloading sample PBIX file...')
    response = requests.get(download_url)
    response.raise_for_status()
    with open(pbi_file, 'wb') as f:
        f.write(response.content)
    print(f'Downloaded: {pbi_file} ({os.path.getsize(pbi_file) / 1024:.0f} KB)')

## 2. Inspect the Semantic Model

`pbixray` parses the PBIX file and exposes all semantic model components
as pandas DataFrames. This section shows what is available before we
generate documentation.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pbixray import PBIXRay

model = PBIXRay(pbi_file)

print('=== Model Summary ===')
print(f'Tables:             {len(model.tables)}')
print(f'Schema columns:     {len(model.schema)}')
print(f'DAX measures:       {len(model.dax_measures)}')
print(f'DAX calc columns:   {len(model.dax_columns)}')
print(f'Power Query steps:  {len(model.power_query)}')
print(f'Relationships:      {len(model.relationships)}')
print(f'M parameters:       {len(model.m_parameters)}')
print(f'File size:          {model.size / (1024*1024):.2f} MB')
print()
print('Tables:', model.tables)

In [ ]:
# Preview DAX measures
print('DAX Measures (first 10):')
model.dax_measures[['TableName', 'Name', 'Expression']].head(10)

In [ ]:
# Preview relationships
print('Relationships:')
model.relationships[['FromTableName', 'FromColumnName', 'ToTableName', 'ToColumnName', 'Cardinality']].head(10)

## 3. Generate Documentation (without LLM explanations)

We run `PBIDocumentGenerator` with `enable_explanation=False` so this works
regardless of whether Ollama is installed. The output is the complete
structural documentation of the model.

All callbacks are replaced with simple print functions so you can see
progress in the notebook output.

In [ ]:
from doc_generator import PBIDocumentGenerator

def print_progress(value):
    bar = '#' * int(value * 30)
    print(f'\r[{bar:<30}] {int(value * 100)}%', end='', flush=True)

docgen = PBIDocumentGenerator(
    pbix_path=pbi_file,
    enable_explanation=False,
    progress_callback=print_progress,
)

print('Generating documentation...')
mdata = docgen.generate()
print()  # newline after progress bar
print(f'Done. Document length: {len(mdata):,} characters')

In [ ]:
# Preview the first section of the generated Markdown
preview_lines = mdata.split('\n')[:60]
print('\n'.join(preview_lines))
print('\n... (truncated)')

## 4. Optional: Add LLM Explanations via Ollama

This section adds plain-English explanations to each DAX measure and
Power Query script using a locally hosted Ollama model.

**Skip this section if Ollama is not installed.** The cell will detect
whether Ollama is reachable before running and skip gracefully if not.

To install Ollama and pull the model:
```bash
# Install from https://ollama.com, then:
ollama pull gemma3:1b
```

In [ ]:
import requests as req

def ollama_is_available():
    try:
        r = req.get('http://localhost:11434/api/tags', timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if not ollama_is_available():
    print('Ollama is not running -- skipping LLM explanations.')
    print('The documentation generated in Section 3 is complete and usable without them.')
    mdata_with_explanations = mdata
else:
    print('Ollama detected. Generating explanations...')
    print('This will take several minutes depending on the number of DAX measures.')
    print()

    streamed_tokens = []

    def stream_to_buffer(gen):
        for token in gen:
            streamed_tokens.append(token)

    def print_scratch(label):
        print(f'  Processing: {label}')

    docgen_llm = PBIDocumentGenerator(
        pbix_path=pbi_file,
        enable_explanation=True,
        scratch_callback=print_scratch,
        clear_scratch_callback=lambda: None,
        progress_callback=print_progress,
        stream_callback=stream_to_buffer,
    )

    mdata_with_explanations = docgen_llm.generate()
    print()
    print(f'Done. Document length: {len(mdata_with_explanations):,} characters')

## 5. Export to HTML

Convert the Markdown document to a styled HTML file and save it locally.
Open `sample_documentation.html` in your browser to review the full output.

In [ ]:
import re
import markdown2

def markdown_to_html(mdata, title='Model Documentation'):
    """Convert Markdown to styled HTML. Mirrors the logic in doc_exporter.py."""
    css = """
    <style>
    body { font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6; margin: 2em; color: #333; }
    h1, h2, h3, h4 { margin-top: 1.5em; font-weight: 600; }
    pre, code { font-family: monospace; background: #f6f8fa; border: 1px solid #e1e4e8;
                border-radius: 4px; padding: 0.2em 0.4em; }
    pre { padding: 1em; overflow-x: auto; }
    table { border-collapse: collapse; margin: 1em 0; width: 100%; table-layout: fixed; }
    th, td { border: 1px solid #dfe2e5; padding: 0.6em; text-align: left;
             vertical-align: top; word-wrap: break-word; }
    th { background: #f6f8fa; font-weight: bold; }
    </style>
    """

    def escape_underscores(text):
        return text.replace('_', r'\_ ')

    mdata = mdata.replace('```dax', '```').replace('```m', '```')
    segments = re.split(r'(```.*?```)', mdata, flags=re.DOTALL)
    processed = []
    for segment in segments:
        if segment.startswith('```'):
            processed.append(segment)
        else:
            sub = re.split(r'(`.*?`)', segment, flags=re.DOTALL)
            processed.append(''.join(s if s.startswith('`') else escape_underscores(s) for s in sub))

    html_body = markdown2.markdown(''.join(processed), extras=['tables'])
    html_body = re.sub(r'<code>\n', '<code>', html_body)
    return f'<html><head><title>{title}</title>{css}</head><body>{html_body}</body></html>'


output_file = 'sample_documentation.html'
html = markdown_to_html(mdata_with_explanations, title=f'{docgen.model_name} - Model Documentation')

with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'HTML documentation written to: {output_file}')
print(f'File size: {os.path.getsize(output_file) / 1024:.0f} KB')
print()
print('Open the file in your browser to review the full documentation.')

## 6. Save as Markdown

The raw Markdown is also saved -- useful for diffing, version control,
or pasting into a documentation platform.

In [ ]:
md_file = 'sample_documentation.md'

with open(md_file, 'w', encoding='utf-8') as f:
    f.write(mdata_with_explanations)

print(f'Markdown documentation written to: {md_file}')
print(f'File size: {os.path.getsize(md_file) / 1024:.0f} KB')

---

## Summary

| Step | What happened |
|---|---|
| Download | Fetched a real Microsoft sample PBIX from a public repo |
| Inspect | Parsed the semantic model with `pbixray` -- tables, measures, relationships |
| Generate | Produced a canonical Markdown document from the full model |
| LLM (optional) | Added plain-English DAX/M explanations via local Ollama |
| Export | Saved both HTML and Markdown output files |

To run against your own PBIX file, replace `pbi_file = 'sample.pbix'` in Section 1
with the path to your file and re-run all cells.

To use the full Streamlit UI with file upload and export buttons:
```bash
streamlit run ../src/app.py
```